# Exercise 4: Barrier vs Nowait Performance Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## 1. Load and Process Results

In [ ]:
# Read CSV output from dmvm_barrier_nowait program
df = pd.read_csv("results.csv", comment="#")

# Clean column names (remove leading spaces)
df.columns = df.columns.str.strip()

# Calculate barrier overhead
df['barrier_overhead_time'] = df['v1_time'] - df['v3_time']
df['barrier_overhead_pct'] = (df['barrier_overhead_time'] / df['v1_time']) * 100

print("Performance Data:")
print(df)
print(f"\nMean barrier overhead: {df['barrier_overhead_pct'].mean():.2f}%")

## 2. CPU Time Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df['threads'], df['v1_time'], 'o-', linewidth=2, markersize=8, 
        label='V1 (implicit barrier)', color='#e74c3c')
ax.plot(df['threads'], df['v3_time'], 's-', linewidth=2, markersize=8,
        label='V3 (static + nowait)', color='#2ecc71')

ax.set_xlabel('Number of Threads', fontsize=12)
ax.set_ylabel('CPU Time (s)', fontsize=12)
ax.set_title('DMVM: Execution Time Comparison', fontsize=14, weight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Annotate improvement
for i, row in df.iterrows():
    if row['threads'] in [4, 8]:  # Annotate key points
        improvement = row['barrier_overhead_pct']
        ax.annotate(f'{improvement:.1f}% faster', 
                   xy=(row['threads'], row['v1_time']),
                   xytext=(10, 10), textcoords='offset points',
                   fontsize=9, color='green',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.7))

plt.tight_layout()
plt.savefig('ex4_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Speedup Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Speedup
ax1.plot(df['threads'], df['v1_speedup'], 'o-', linewidth=2, markersize=8,
         label='V1 (with barrier)', color='#e74c3c')
ax1.plot(df['threads'], df['v3_speedup'], 's-', linewidth=2, markersize=8,
         label='V3 (nowait)', color='#2ecc71')
ax1.plot(df['threads'], df['threads'], '--', linewidth=2, color='gray', alpha=0.5,
         label='Ideal (Linear)')

ax1.set_xlabel('Number of Threads', fontsize=12)
ax1.set_ylabel('Speedup', fontsize=12)
ax1.set_title('Speedup Comparison', fontsize=14, weight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Efficiency
ax2.plot(df['threads'], df['v1_eff'] * 100, 'o-', linewidth=2, markersize=8,
         label='V1 (with barrier)', color='#e74c3c')
ax2.plot(df['threads'], df['v3_eff'] * 100, 's-', linewidth=2, markersize=8,
         label='V3 (nowait)', color='#2ecc71')
ax2.axhline(y=100, linestyle='--', linewidth=2, color='gray', alpha=0.5,
            label='Ideal (100%)')

ax2.set_xlabel('Number of Threads', fontsize=12)
ax2.set_ylabel('Efficiency (%)', fontsize=12)
ax2.set_title('Parallel Efficiency', fontsize=14, weight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('ex4_speedup_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. MFLOPS Performance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df['threads'], df['v1_mflops'], 'o-', linewidth=2, markersize=8,
        label='V1 (implicit barrier)', color='#e74c3c')
ax.plot(df['threads'], df['v3_mflops'], 's-', linewidth=2, markersize=8,
        label='V3 (nowait)', color='#2ecc71')

ax.set_xlabel('Number of Threads', fontsize=12)
ax.set_ylabel('MFLOP/s', fontsize=12)
ax.set_title('Computational Throughput', fontsize=14, weight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Add peak MFLOPS annotation
max_mflops = df['v3_mflops'].max()
max_threads = df.loc[df['v3_mflops'].idxmax(), 'threads']
ax.annotate(f'Peak: {max_mflops:.1f} MFLOP/s\n@ {int(max_threads)} threads',
            xy=(max_threads, max_mflops),
            xytext=(20, -30), textcoords='offset points',
            fontsize=10, weight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgreen', alpha=0.8),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', lw=2))

plt.tight_layout()
plt.savefig('ex4_mflops.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Barrier Overhead Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Absolute overhead
ax1.bar(df['threads'], df['barrier_overhead_time'] * 1000, 
        color='#f39c12', edgecolor='black', linewidth=1.5, alpha=0.8)
ax1.set_xlabel('Number of Threads', fontsize=12)
ax1.set_ylabel('Barrier Overhead (ms)', fontsize=12)
ax1.set_title('Absolute Barrier Overhead', fontsize=14, weight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for i, (th, overhead) in enumerate(zip(df['threads'], df['barrier_overhead_time'])):
    ax1.text(th, overhead * 1000, f'{overhead*1000:.2f} ms',
            ha='center', va='bottom', fontsize=9)

# Relative overhead
ax2.bar(df['threads'], df['barrier_overhead_pct'],
        color='#e74c3c', edgecolor='black', linewidth=1.5, alpha=0.8)
ax2.set_xlabel('Number of Threads', fontsize=12)
ax2.set_ylabel('Barrier Overhead (%)', fontsize=12)
ax2.set_title('Relative Barrier Overhead', fontsize=14, weight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, (th, overhead) in enumerate(zip(df['threads'], df['barrier_overhead_pct'])):
    ax2.text(th, overhead, f'{overhead:.2f}%',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('ex4_overhead.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBarrier Overhead Statistics:")
print(f"  Min: {df['barrier_overhead_pct'].min():.2f}%")
print(f"  Max: {df['barrier_overhead_pct'].max():.2f}%")
print(f"  Mean: {df['barrier_overhead_pct'].mean():.2f}%")

## 6. Scalability Summary Table

In [ ]:
summary = df[['threads', 'v1_time', 'v1_speedup', 'v1_eff', 
              'v3_time', 'v3_speedup', 'v3_eff', 'barrier_overhead_pct']].copy()
summary.columns = ['Threads', 'V1_Time(s)', 'V1_Speedup', 'V1_Eff', 
                   'V3_Time(s)', 'V3_Speedup', 'V3_Eff', 'Overhead(%)']

print("\nPerformance Summary:")
print(summary.to_string(index=False))

# Create formatted table visualization
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')

table_data = []
for _, row in summary.iterrows():
    table_data.append([
        int(row['Threads']),
        f"{row['V1_Time(s)']:.4f}",
        f"{row['V1_Speedup']:.2f}x",
        f"{row['V1_Eff']:.2%}",
        f"{row['V3_Time(s)']:.4f}",
        f"{row['V3_Speedup']:.2f}x",
        f"{row['V3_Eff']:.2%}",
        f"{row['Overhead(%)']:.2f}%"
    ])

table = ax.table(cellText=table_data,
                colLabels=['Threads', 'V1 Time', 'V1 Speedup', 'V1 Eff',
                          'V3 Time', 'V3 Speedup', 'V3 Eff', 'Overhead'],
                cellLoc='center',
                loc='center',
                colWidths=[0.1, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Color header
for i in range(8):
    table[(0, i)].set_facecolor('#3498db')
    table[(0, i)].set_text_props(weight='bold', color='white')

plt.title('DMVM Performance Summary', fontsize=14, weight='bold', pad=20)
plt.savefig('ex4_table.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Takeaways

### Observations:
1. **Barrier overhead**: Typically 1-5% of execution time
2. **Nowait benefit**: More pronounced with higher thread counts
3. **Scalability**: Near-linear speedup up to physical cores

### Best Practices:
- Use `nowait` when results aren't immediately needed
- Add explicit barrier only when necessary
- Profile to measure actual barrier cost
- Consider memory bandwidth as limiting factor

### When Barriers Matter:
- Fine-grained parallelism (many small loops)
- Iterative algorithms (barriers in loops)
- High thread counts (barrier cost increases with threads)